# Differentiation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caam37830-sci-comp/notes/blob/main/04_functions/differentiation.ipynb)


In [ ]:
import numpy as np
from numpy import exp
import scipy.sparse as sparse
import matplotlib.pyplot as plt

# Numerical differentiation

## Finite differences
The simplest way to numerically approximate a derivative is by approximating the above limit. This is called a "finite difference method".

The three simplest finite methods are the **forward difference**
$$ f'(x) \approx \frac{f(x+h) - f(x)}{h}$$
the **backwards difference**
$$ f'(x) \approx \frac{f(x) - f(x-h)}{h}$$
and the **central difference**
$$ f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

In [ ]:
def forward_diff(f,x,h=1e-8):
    return (f(x+h)-f(x))/h

def backward_diff(f,x,h=1e-8):
    return (f(x)-f(x-h))/h

def central_diff(f,x,h=1e-8):
    return (f(x+h)-f(x-h))/h/2.

g = lambda x: exp(-x)*x*(x-1)
gp = lambda x: exp(-x)*(-x*(x-1)+(x-1)+x)

x0 = 0.2
print("True value : {}".format(gp(x0)))
for diff in [forward_diff, backward_diff, central_diff]:
    print("{} : {}".format(diff.__name__, diff(g,x0,1e-3)))

In [ ]:
hs = np.logspace(0, -16)
errs = np.zeros(len(hs))
for i in range(len(hs)):
    errs[i] = abs(central_diff(g,x0,hs[i]) - gp(x0))
    
plt.loglog(hs,errs)

Let's look at the errors

In [ ]:
for diff in [forward_diff, backward_diff, central_diff]:
    print("Error in {} : {}".format(diff.__name__, abs(diff(g,x0,1e-5)-gp(x0))))

We see the central difference gives a better approximation. This is due to the fact that when $f$ is smooth, we have that
$$ f'(x) = \frac{f(x+h) - f(x)}{h} + O(h)$$

$$ f'(x) \approx \frac{f(x) - f(x-h)}{h} + O(h)$$

$$ f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} + O(h^2)$$

# Numerical Differentiation on a Grid
Suppose we have a function on a grid

In [ ]:
g = lambda x: np.sin(3*x)*exp(-x*x)

a, b = -5, 5
n = 1000

xs = np.linspace(a, b, n) # start from a, end at b, n points
h = (b-a) / (n-1)
gx = g(xs)

What if we want to evaluate the derivative at each point on the grid?

`numpy` provides a function to efficiently compute this using differences: `diff`. This is equivalent to

    np.array([gx[i+1] - gx[i] for i in range(len(gx) - 1)

In order to approximate the derivative using a forward difference, we simply divide by $h$. Note that we can't use the forward difference to approximate value at the last grid point.

In [ ]:
import matplotlib.pyplot as plt
g1 = np.diff(gx)/h
plt.plot(xs, gx, label= 'g')
plt.plot(xs[:-1], g1, label="g'") # plot must exclude end point
plt.legend()
plt.show()

Let $\{ x_0, x_1, \dots x_{n-1}\}$ be the grid of points with mesh $|x_{i+1} - x_{i}| = h$. The continuous function $f$ defined on the grid is an $n$-vector
$$ f_{\mathrm {vec}} = \left( f(x_0), \dots f(x_{n-1})  \right)^\top.$$

The forward differnce operator can be expressed as applying the sparse matrix
$$ A = \frac1h \begin{pmatrix} -1 & 1 &&0 \\  & -1 & 1 &\\ & &-1 &1 & \\ 0&&& \ddots\end{pmatrix}$$
which could be implemented using the `linearoperator` class or the sparse matrix classes from earlier lectures.
So that $Af_{\mathrm {vec}}$ is the finite difference on $f$.

In [ ]:
import scipy.sparse as sparse

In [ ]:
# create the forward difference matrix A, equivalent to np.diff
def forward_diff_matrix(n):
    data = []
    i = []
    j = []
    for k in range(n - 1):
        i.append(k)
        j.append(k)
        data.append(-1)
        
        i.append(k)
        j.append(k+1)
        data.append(1)
        
    # we'll just set the last entry to 0 to have a square matrix
    return sparse.coo_matrix((data,(i,j)),shape=(n,n)).tocsr()

D = forward_diff_matrix(len(gx))/h
D.toarray()

In [ ]:
gx1 = D*gx
plt.plot(xs,gx1)
plt.show()

Note that difference schemes can't compute the derivate at the two end points.

One approach is to use 

    * Central difference in the interior
    * Forward difference at the left
    * Back difference at the right

This can be expressed as applying the sparse matrix
$$ A = \frac1h \begin{pmatrix} -1 & 1 & \cdots& &0 \\  -1/2& 0 & 1/2 &\\ & -1/2 & 0 & 1/2 \\ 0&&& \ddots \\ &&&-1 & 1  \end{pmatrix}$$

In [ ]:
# create the forward difference matrix A, equivalent to np.diff
def diff_matrix(n):
    data = []
    i = []
    j = []
    
    # forward on the left
    i.append(0)
    j.append(0)
    data.append(-1)
    
    i.append(0)
    j.append(1)
    data.append(1)
    
    # central in the middle
    for k in range(1,n - 1):
        i.append(k)
        j.append(k-1)
        data.append(-0.5)
        
        i.append(k)
        j.append(k+1)
        data.append(0.5)
        
    # backward on the right
    # forward on the left
    i.append(n-1)
    j.append(n-1)
    data.append(1)
    
    i.append(n-1)
    j.append(n-2)
    data.append(-1)
        
    return sparse.coo_matrix((data,(i,j)),shape=(n,n)).tocsr()

D = diff_matrix(len(gx))/h
D.toarray()

In [ ]:
gx1 = D*gx
plt.plot(xs,gx1)
plt.show()

There are also more complicated difference formulas. We could, for example, achieve more accuracy on the right using the so-called BDF2 formula

$$ f'(x) \approx \frac{f(x) - \frac43 f(x-h) + \frac13 f(x-2h)}{\frac23 h}+O(h^2)$$

Finite difference schemes are designed by choosing a "stencil" (the required function values) and then finding the coefficients that work through Taylor series calculations.

# Higher order derivatives
For simplicity, we'll use the square matrix above

In [ ]:
D1 = diff_matrix(len(gx))
plt.spy(D1)

Because differentiation is linear, we can compute second derivatives by repeatedly applying the above derivative operator

In [ ]:
# second derivative
D2 = D1 @ D1

In [ ]:
# third derivative
D3 = D1 @ D2

In [ ]:
gx2 = D2 @ gx
plt.plot(xs,gx2)
plt.show()

In [ ]:
D2[:8,:8].todense()

What we are seeing here is that repeatedly applying the central difference gives

$$ f''(x)\approx \frac{f(x+2h)-2 f(x) + f(x-2h)}{4h^2} $$

If we applied forward and then backwards difference, we would find that

$$ f''(x)\approx \frac{f(x+h)-2 f(x) + f(x-h)}{h^2} $$

# Optional: Higher dimensional grids

In [ ]:
xx, yy = np.meshgrid(np.linspace(-10,10,100),np.linspace(-10,10,100))
g2d = np.sin(xx) + np.cos(yy)
plt.imshow(g2d)
plt.show()

Recall that a derivative measures the rate of change of a function
$$ \frac{df}{dx} = \lim_{h\to 0} \frac{f(x+h)-f(x)}{h}$$
For a functions of multiple variables we also have the partial derivative
$$ \frac{\partial f}{\partial x_i} = \lim_{h\to 0} \frac{f(x+h e_i)-f(x)}{h}$$
the **gradient**
$$ \nabla f = \begin{bmatrix} \frac{\partial f}{\partial x_1} ,\frac{\partial f}{\partial x_2},\ldots\end{bmatrix}^T$$
and the **Hessian**
$$ H = \left[\frac{\partial^2 f}{\partial x_i\partial x_j} \right]_{i,j}$$

We will create an operator that will act as a derivative on the function treated as a vector

In [ ]:
# turn 2-dimensional function to a vector
g = g2d.flatten()

Recall that numpy arrays are row-major by default. This means that if we want to compute derivatives in the $x$ direction, we can simply create a block matrix with a derivative operator for each block in a contiguous row.

`scipy.sparse.kron` is a very useful function to create such a sparse matrix.

The Kronecker product $A\otimes B$ is the block matrix whose $(i,j)$ block is $a_{ij}B$. If $D$ is the 1-dimensional difference matrix from before, then

* $D_x = I\otimes D$ is block diagonal. It applies $D$ to each row of the grid, i.e. to each contiguous block of the vector.
* $D_y = D\otimes I$ takes differences between rows of the grid, i.e. between entries that are $n_x$ apart in the vector.

Here are the sparsity patterns on a $4\times 4$ grid

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].spy(sparse.kron(sparse.eye(4), diff_matrix(4)))
axs[0].set_title(r'$I \otimes D$')
axs[1].spy(sparse.kron(diff_matrix(4), sparse.eye(4)))
axs[1].set_title(r'$D \otimes I$')
plt.show()

Now we build the operators for our $100\times 100$ grid. Note that `meshgrid` puts $x$ along the columns and $y$ along the rows.

In [ ]:
ny, nx = g2d.shape
hx = xx[0, 1] - xx[0, 0] # x changes along each row
hy = yy[1, 0] - yy[0, 0] # y changes down each column

Dx = sparse.kron(sparse.eye(ny), diff_matrix(nx)).tocsr() / hx
Dy = sparse.kron(diff_matrix(ny), sparse.eye(nx)).tocsr() / hy

Applying an operator to the vector and reshaping the result gives the partial derivative on the grid

In [ ]:
dgdx = (Dx @ g).reshape(g2d.shape)
dgdy = (Dy @ g).reshape(g2d.shape)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].imshow(dgdx)
axs[0].set_title(r'$\partial g / \partial x$')
axs[1].imshow(dgdy)
axs[1].set_title(r'$\partial g / \partial y$')
plt.show()

Since $g = \sin x + \cos y$, we know $\partial_x g = \cos x$ and $\partial_y g = -\sin y$. As in one dimension, the error is $O(h^2)$ in the interior and $O(h)$ on the boundary, where we fall back to one-sided differences.

In [ ]:
errx = np.abs(dgdx - np.cos(xx))
erry = np.abs(dgdy + np.sin(yy))
print("max error in x            :", errx.max())
print("max error in x (interior) :", errx[:, 1:-1].max())
print("max error in y            :", erry.max())
print("max error in y (interior) :", erry[1:-1, :].max())

`numpy` also provides `np.gradient`, which uses the same scheme (central differences in the interior, one-sided differences at the ends) along every axis of an array. It returns one array per axis, so for our grid the first output is the $y$-derivative.

In [ ]:
dgdy_np, dgdx_np = np.gradient(g2d, hy, hx)
print(np.allclose(dgdx, dgdx_np), np.allclose(dgdy, dgdy_np))

With $D_x$ and $D_y$ we can assemble these as sparse operators on the flattened grid function. With $N = n_x n_y$ grid points, the gradient stacks the two partials into a $2N\times N$ matrix
$$ \nabla \approx \begin{bmatrix} D_x \\ D_y \end{bmatrix}$$
and the entries of the Hessian are approximated by $D_x D_x$, $D_x D_y$, $D_y D_x$ and $D_y D_y$. Because $D_x$ and $D_y$ act on different indices,
$$ (I\otimes D)(D\otimes I) = D\otimes D = (D\otimes I)(I\otimes D)$$
so the discrete mixed partials are equal, just like the continuous ones.

In [ ]:
grad = sparse.vstack([Dx, Dy])
print(grad.shape)

Dxx = Dx @ Dx
Dxy = Dx @ Dy
Dyy = Dy @ Dy
print("max |DxDy - DyDx| :", abs(Dx @ Dy - Dy @ Dx).max())

The **Laplacian** is the trace of the Hessian
$$ \Delta f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}$$
For our function, $\Delta g = -\sin x - \cos y = -g$.

In [ ]:
L = Dxx + Dyy
Lg = (L @ g).reshape(g2d.shape)

err = np.abs(Lg + g2d)
print("max error          :", err.max())
print("max interior error :", err[2:-2, 2:-2].max())
plt.imshow(err)
plt.colorbar()
plt.show()